In [1]:
%pip install trl
%pip install -U trackio

import sys
!{sys.executable} -m pip install --upgrade --no-cache-dir "jinja2>=3.1.6"

from importlib.metadata import version
print(version("jinja2"))

import jinja2
print(jinja2.__version__)

# %conda install -y -c conda-forge compilers
!which gcc
!which clang

import os
os.environ["CC"] = "/opt/conda/bin/gcc"
! /opt/conda/bin/gcc --version




Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
3.1.6
3.1.6
/opt/conda/bin/gcc
gcc (conda-forge gcc 14.2.0-2) 14.2.0
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [2]:
import torch
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer

use_cuda = torch.cuda.is_available()
print(f"GPU available: {torch.cuda.is_available()}")

!nvidia-smi

GPU available: True
Wed Nov 25 01:19:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.108                Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1050        On  |   00000000:01:00.0  On |                  N/A |
| N/A   41C    P3            N/A  / 5001W |    1471MiB /   3072MiB |      4%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [3]:
from huggingface_hub import notebook_login

notebook_login()


In [4]:
from transformers import TrainerCallback

# https://huggingface.co/docs/transformers/en/main_classes/callback
class LoggerCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return

        row = {"step": state.global_step}
        for key, value in logs.items():
            if isinstance(value, torch.Tensor):
                value = value.detach().cpu().item()
            row[key] = value

        os.makedirs(os.path.dirname(LOG_CSV), exist_ok=True)

        new_row = pd.DataFrame([row])

        if os.path.exists(LOG_CSV):
            old_df = pd.read_csv(LOG_CSV)
            # Conserve les colonnes même si un log ne contient pas toutes les métriques.
            all_columns = list(dict.fromkeys(list(old_df.columns) + list(new_row.columns)))
            combined = pd.concat(
                [old_df.reindex(columns=all_columns),
                 new_row.reindex(columns=all_columns)],
                ignore_index=True,
            )
        else:
            combined = new_row

        combined.to_csv(LOG_CSV, index=False)

In [5]:
import os
import pandas as pd
import torch
from transformers import TrainerCallback
from trl import DPOConfig, DPOTrainer

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
OUTPUT_MODEL = f"./{MODEL}_dpo"
LOG_CSV = os.path.join(OUTPUT_MODEL, "training_metrics.csv")


use_cuda = torch.cuda.is_available()

# https://huggingface.co/datasets/trl-lib/ultrafeedback_binarized
dataset = load_dataset(
    "trl-lib/ultrafeedback_binarized",
    split="train",
)
small_dataset = dataset.select(range(min(5000, len(dataset))))

# https://huggingface.co/docs/trl/dpo_trainer
# https://walkinglabs.github.io/hands-on-modern-rl/en/chapter17_dpo/metrics
args = DPOConfig(
    output_dir=OUTPUT_MODEL,
    bf16=False,
    fp16=use_cuda,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    max_length=128,
    logging_steps=10,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    report_to="trackio",
    use_cpu=not use_cuda,
)

trainer = DPOTrainer(
    model=MODEL,
    train_dataset=small_dataset,
    args=args,
    callbacks=[LoggerCallback()],
)

trainer.train()
trainer.save_model(OUTPUT_MODEL)

print(f"Métriques enregistrées dans : {LOG_CSV}")





Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

* Trackio project initialized: huggingface
* Trackio metrics logged to: /root/.cache/huggingface/trackio
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: happy-ocean-6


Step,Training Loss
10,0.692533
20,0.694073
30,0.694273
40,0.693412
50,0.695078
60,0.691989
70,0.691763
80,0.695152
90,0.691894
100,0.691587


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:
# Fonctionne pas
# import trackio
# trackio.show()

# https://huggingface.co/docs/trackio/en/index

import os
import pandas as pd
import matplotlib.pyplot as plt

results_dir = os.path.join(OUTPUT_MODEL, "results")
os.makedirs(results_dir, exist_ok=True)

logs = pd.DataFrame(trainer.state.log_history)
logs.to_csv(
    os.path.join(results_dir, "training_metrics.csv"),
    index=False,
)

metrics = [
    col for col in logs.select_dtypes(include="number").columns
    if col != "step"
]

for metric in metrics:
    data = logs.dropna(subset=[metric])

    if not data.empty and "step" in data.columns:
        plt.figure(figsize=(8, 4))
        plt.plot(data["step"], data[metric], marker=".")
        plt.title(metric)
        plt.xlabel("Étape")
        plt.ylabel(metric)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(
            os.path.join(results_dir, f"{metric.replace('/', '_')}.png"),
            dpi=150,
        )
        plt.close()

print("CSV et graphiques enregistrés dans :", results_dir)





CSV et graphiques enregistrés dans : ./HuggingFaceTB/SmolLM2-135M-Instruct_dpo/results


In [7]:
print("Done")

Done
